# SpherIDiff at Band Limit 31

This notebook pre-trains the four SpherIDiff sizes with band limit L = 31 instead of L = 16, and evaluates them under the fixed final protocol. With L = 16, the band limit of the 32 x 64 grid under Clenshaw-Curtis quadrature, the band-limited target leaves a large part of the field out of reach. Raising the band limit to L = 31 lowers this floor of the normalised RMSE, measured on one training year, from about 0.27 to about 0.10.

Three amendments are used with L = 31 within a small parameter budget. They are set in the model configurations `spheridiff_l31_xs`, `spheridiff_l31_s`, `spheridiff_l31_m` and `spheridiff_l31_b4`, and the earlier configurations are unchanged.

- **Least-squares analysis.** The data grid is equiangular with poles, where quadrature supports only L = 16. The transform at L = 31 (K = 961 coefficients) is fitted by weighted least squares. It is well conditioned on this grid, with a condition number of about 2.4. The input is band-limited once with it, and the spectral noise is drawn with the same K.
- **Diagonal plus low-rank spectral weight.** Each block uses W = D + P Q^T, with D diagonal and P, Q of rank r = 32. The weight is still a full-rank K x K matrix, so the Slepian projection and the low-rank update for regional fine-tuning apply to it unchanged.
- **Internal Gauss grid.** The blocks run on a 32 x 64 Legendre-Gauss grid, where quadrature is exact up to L = 32. The transform on this grid uses dense matrices. A separable form (a Fourier transform along longitude followed by a Legendre contraction for each order) gives the same result, but it was about 20% slower per step in a local test at this grid size, so it is not used. The input is moved onto this grid once at the start and the output back onto the data grid at the end, both by exact resampling of band-limited fields.

| Run | Widths | Parameters |
|---|---|---|
| `pilot_spheridiff_l31_xs` | 64, 128, 64 | 264,987 |
| `pilot_spheridiff_l31_s` | 32, 64, 64, 64, 32 | 335,901 |
| `pilot_spheridiff_l31_m` | 64, 128, 128, 128, 64 | 587,805 |
| `pilot_spheridiff_l31_b4` | 128, 256, 256, 256, 128 | 1,589,277 |

Every run is trained for 25,000 steps (`pilot_25k`), the fixed comparison budget. The other training settings are those of the pilot, and the seed is 42.

The notebook uses the ERA5 cache at 5.6 degrees, attached as a Kaggle input, and the repository code, cloned from GitHub. It is designed for a background run on two T4 GPUs. The results are packed into `pilot_l31_results.zip` in the working folder.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
FIRST_GPU_EXPERIMENTS = ["pilot_spheridiff_l31_xs", "pilot_spheridiff_l31_b4"]
SECOND_GPU_EXPERIMENTS = ["pilot_spheridiff_l31_s", "pilot_spheridiff_l31_m"]
RUN_NAMES = FIRST_GPU_EXPERIMENTS + SECOND_GPU_EXPERIMENTS
REFERENCE_RUNS = ["pilot_wind_s", "pilot_wind_m", "pilot_wind_l", "pilot_spheridiff_b4"]
ARCHIVE = WORK_ROOT / "pilot_l31_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. The data cache is read from the attached dataset `thesyeddaniyal/era5-pilot-6degrees`. The folder that holds `stats.npz` is used as the cache root. When the expected input path does not exist, the whole of `/kaggle/input` is searched for `stats.npz`.

The runs of the main pilot (`pilot_wind_s`, `pilot_wind_m`, `pilot_wind_l` and `pilot_spheridiff_b4` at L = 16) serve as references when they are attached as a Kaggle input. They are found by the folder name and the `evaluations.jsonl` file inside. Without them, the runs are trained and stored, and the comparison is left out. The references were trained for 45,000 steps, so the comparison of the final errors favours them. Their errors at 25,000 steps are shown in the evaluation curves and listed among the errors at fixed fractions of training.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
DATASET_ROOT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


def locate_reference(name):
    for root in (INPUT_ROOT, WORK_ROOT):
        for current, _, files in os.walk(root):
            if Path(current).name == name and "evaluations.jsonl" in files:
                return Path(current).parent
    return None


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
CACHE_ROOT = locate("stats.npz", DATASET_ROOT if DATASET_ROOT.exists() else INPUT_ROOT)
REFERENCE_ROOT = locate_reference(REFERENCE_RUNS[0])
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT)
print("reference runs:", REFERENCE_ROOT)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

Each GPU trains one run at a time, so that two pre-trainings run in parallel without sharing a card. The runs are paired to balance the work of the two cards: the first GPU trains `xs` and then `b4`, the smallest and the largest run, and the second GPU trains `s` and then `m`. The smaller run of each pair comes first, so its results are available early. With one GPU, both queues share it.

Each job is a separate process with its own log file in `runs/_logs`. The notebook prints every line that a job writes as soon as it appears, prefixed with the elapsed time, the GPU and the job name. The same lines are kept in `runs/_logs/parallel.log`. During training, every run is evaluated every 1,250 steps on 32 fixed validation windows (validation loss, normalised RMSE and CRPS), and checkpoints at 25%, 50% and 75% of the steps are kept.

The jobs and the packing of the results share one cell with a `try` and a `finally`. When a job fails, or the cell is interrupted, the figures, the status file and the archive are still produced. The archive is also refreshed every 30 minutes during the run. When all training jobs have ended, the final evaluation runs on the same two queues. It evaluates every run under the fixed protocol `final_v1` on the held-out test year: the forecast error and CRPS, and the six reconstruction tasks, each solved with MMPS guidance and with two forms of native denoising. It writes `final_evaluation.json` into each run folder, and the table `final_evaluation_summary.txt` lists all runs under the same protocol.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import final_job, place_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
LOGS = RUNS_ROOT / "_logs"
TRAINING_CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
EVALUATION_CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
TRAINING_PLACEMENT = [
    (0, [train_job(name) for name in FIRST_GPU_EXPERIMENTS]),
    (1, [train_job(name) for name in SECOND_GPU_EXPERIMENTS]),
]
EVALUATION_PLACEMENT = [
    (0, [final_job(name, FINAL_PROTOCOL) for name in FIRST_GPU_EXPERIMENTS]),
    (1, [final_job(name, FINAL_PROTOCOL) for name in SECOND_GPU_EXPERIMENTS]),
]
COMPARISONS = [(run, reference) for run in RUN_NAMES for reference in REFERENCE_RUNS] if REFERENCE_ROOT else []


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
EVALUATOR = None
try:
    RUNNER = ParallelRunner(place_queues(TRAINING_PLACEMENT, GPUS), TRAINING_CONTEXT, refresh=refresh)
    RUNNER.run()
    EVALUATOR = ParallelRunner(place_queues(EVALUATION_PLACEMENT, GPUS), EVALUATION_CONTEXT, refresh=refresh)
    EVALUATOR.run()
finally:
    OUTCOMES = [*(RUNNER.outcomes() if RUNNER else []), *(EVALUATOR.outcomes() if EVALUATOR else [])]
    finalise(
        RUNS_ROOT,
        ARCHIVE,
        OUTCOMES,
        RUN_NAMES,
        comparisons=COMPARISONS,
        reference_root=REFERENCE_ROOT,
        at_final=True,
        final_names=RUN_NAMES,
        final_reference_names=REFERENCE_RUNS if REFERENCE_ROOT else [],
    )
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summaries list the steps, training time, time per step, peak memory and final loss of every run. When the reference runs are available, the comparison summary lists, for each run against each reference, the step at which it reaches the final error of the reference and stays there, the errors at fixed fractions of training, the mean error over the whole curve, and the paired difference of the final errors with a 95% interval over the evaluation windows. The figures show the runs together with the references, against steps, samples seen and training time. The final evaluation table lists, for each run, the number of parameters, the trained steps, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row.

In [ ]:
import json

for name in RUN_NAMES:
    summary = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

comparison = RUNS_ROOT / "comparison_summary.txt"
if comparison.exists():
    print(comparison.read_text(encoding="utf-8"))

final_summary = RUNS_ROOT / "final_evaluation_summary.txt"
if final_summary.exists():
    print(final_summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))